# 32 · Deploying the server: containers, config and GPU scheduling
Level L5 · Part 5 · core · ~20 min · builds on lessons 08, 31

Time: about 20 min. This is a **walkthrough**: you run each step in a terminal or notebook cell and check what you see. Step 3 starts the **real vLLM server** from a pinned image, in Docker Compose, on port **8320**. It takes 1 to 3 minutes on a 4-core CPU when the model is cached from lesson 08. Without Docker, that step shows the **recorded output** (`data/recorded/32.json`), labelled as such.

Kubernetes is not run in this course (it is its own course). Step 4 and Step 5 read a Kubernetes manifest and check it **offline** with `kubeconform`. Nothing is deployed to a cluster.

**To run the Docker steps**, start the notebook with the tags enabled: `TUT_ENABLE=docker,download,slow` (the `download` tag covers the first model pull). Without them, each Docker step prints the recorded output instead.

## Goal
By the end you can package the Brightside server as a pinned Compose service, read the readiness check that tells traffic when the server can answer, name the placement rules a GPU pod needs, and check a manifest offline before it reaches a cluster.

**Terms used here.**
- **Image:** the packaged server: code, libraries and the engine, at one fixed version.
- **Compose service:** one container, defined in a YAML file and started with `docker compose`.
- **Readiness:** the check that says a replica may take traffic. A vLLM server answers `/health` with 200 only after its weights load.
- **GPU scheduling:** the rules that decide which node a pod runs on: the GPU type, the GPU count and the taints and tolerations.
- **Taint and toleration:** a taint marks a node so that pods are kept off it; a toleration on a pod lets that pod land there anyway.
- **Probe:** a check the platform runs on a container. A **startup** probe waits for slow starts, a **readiness** probe gates traffic, and a **liveness** probe restarts a container that stops answering.
- **Spot VM:** a cheaper cloud VM that the provider can reclaim at any time.

## Why it matters
- **Without it:** the server is a command in your shell history. A laptop, a GPU VM and a cluster each get a hand-edited copy of the flags, the port, the model name and the health path. A typo in one copy shows up as a 500 error in production, not at build time.
- **With it:** one pinned image, one config file per environment, a readiness check the platform polls, and placement rules the scheduler enforces. A change is a diff, and a bad change fails in a check.
- **The cost of skipping the readiness check:** a replica that receives traffic while its weights load answers with errors. Lesson 31 showed that a new replica is useless until it is ready.
- **When you don't need it:** one person testing one model on one laptop. `docker run` (lesson 08) is enough until several callers depend on the server.

The next cell counts the hand edits in the "without" version. The numbers are a stand-in for a real team, not a measurement.

In [1]:
settings = ["image tag", "model name", "port", "max model length", "API key source", "health path", "GPU type", "shared memory"]
environments = ["laptop", "GPU VM", "cluster"]
releases_per_quarter = 6
print(f"hand edits per release, without a file: {len(settings) * len(environments)}")
print(f"hand edits per quarter: {len(settings) * len(environments) * releases_per_quarter}")
print("with one config file per environment: the edits are the environment values only, and a check runs on each diff")

hand edits per release, without a file: 24
hand edits per quarter: 144
with one config file per environment: the edits are the environment values only, and a check runs on each diff


## The idea
Packaging a model server means four jobs. **Package** it at one pinned version, so the same bytes run everywhere. **Configure** it with arguments and environment variables, never with secrets written into files. **Tell the platform when it is ready**, with a health check that the platform polls before it sends traffic. **Place** it on the right hardware: a GPU pod names the GPU type and the GPU count, and the scheduler matches it with a node that has one free. The picture is a shipping container: the box is sealed and labelled (the image), the label says what is inside (the config), the dock does not open it until the inspector signs (the readiness check), and the crane puts it on a ship with the right cargo space (the GPU placement).

## Picture
```mermaid
flowchart LR
  I["Pinned image: vllm-openai-cpu v0.32.0"] --> C["Config: arguments and environment, no secrets in the file"]
  C --> P["Start: weights load, minutes"]
  P --> H["Readiness: /health answers 200"]
  H --> T["Traffic from the gateway"]
  G["GPU placement: node label, nvidia.com/gpu, toleration"] --> P
```

## Step by step

### Step 1: the packaged server (Compose file)
The file is `../infra/compose/serve-32.yml`. Its image is the pinned CPU image from lesson 08, `vllm/vllm-openai-cpu:v0.32.0`. The same flags run here: float32 weights for a CPU, eager mode, a 4096-token window, 4 requests in a batch and a 2 GiB KV cache (lessons 03, 16). The port is `127.0.0.1:8320`, so only this machine can reach it. The file has no secret.

Read the file first. It is short (about 30 lines) and holds no secret, so the whole file prints. The two parts to look at are `command` (the server's arguments) and `healthcheck` (the readiness check Docker runs).

In [2]:
import json, os, shutil, subprocess, time
from pathlib import Path
print("\n".join(f"{i:>2} {line}" for i, line in enumerate(Path("../infra/compose/serve-32.yml").read_text().splitlines(), 1)))

 1 # Lesson 32: the vLLM server packaged as one Compose service. Same pinned CPU image and flags as lesson 08.
 2 # Run from notebooks/: docker compose -p tut-kv-forge-15h-32 -f ../infra/compose/serve-32.yml up -d
 3 # No secret is in this file. The API key, if you set one, is passed at run time (lesson 16), never written here.
 4 # SERVE_MODEL overrides the model with one shell variable; the default is the Brightside model from lesson 08.
 5 services:
 6   serve:
 7     image: vllm/vllm-openai-cpu:v0.32.0
 8     command:
 9       - "${SERVE_MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
10       - "--served-model-name"
11       - "brightside"
12       - "--host"
13       - "0.0.0.0"
14       - "--dtype"
15       - "float32"
16       - "--enforce-eager"
17       - "--max-model-len"
18       - "4096"
19       - "--max-num-seqs"
20       - "4"
21       - "--kv-cache-memory-bytes"
22       - "2147483648"
23     ports:
24       - "127.0.0.1:8320:8000"
25     shm_size: "4g"
26     restart: "no"
27    

Now ask Docker to read the file. `docker compose config` resolves the file and checks it. It does not start anything, so it is quick. The cell prints only the fields that matter here. It does not print the whole resolved file: that output shows every value, and a value you add later could be a secret.

In [3]:
COMPOSE = Path("../infra/compose/serve-32.yml").resolve()
STACK = "tut-kv-forge-15h-32"
ENV_FILE = ["--env-file", "../.env"] if Path("../.env").exists() else []     # the run-time settings file, if you have one
COMPOSE_ARGS = ["docker", "compose", "-p", STACK, "-f", str(COMPOSE), *ENV_FILE]
res = subprocess.run([*COMPOSE_ARGS, "config", "--format", "json"], capture_output=True, text=True, cwd=Path(".").resolve())
if res.returncode != 0:
    raise RuntimeError("compose config failed: " + res.stderr[-300:])
svc = json.loads(res.stdout)["services"]["serve"]
live_config = True
print("image:", svc["image"])
print("ports:", [(p["host_ip"], p["published"], p["target"]) for p in svc["ports"]])
shm_bytes = int(svc['shm_size'])     # Compose reports the size as a string of bytes
print(f"shm_size: {shm_bytes} bytes = {shm_bytes / 2**30:.0f} GiB")
print("first arguments:", svc["command"][:3])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


If Docker is not running here, the cell above is skipped and this one shows the recorded output of the same command.

In [4]:
REC = json.loads(Path("../data/recorded/32.json").read_text())
if not globals().get("live_config", False):
    print("recorded output (from the course machine, not this run):")
    print(json.dumps(REC["config"], indent=1))
    print(f"shm_size in the recording: {REC['config']['shm_size']} bytes = {REC['config']['shm_size'] / 2**30:.0f} GiB")
else:
    print("live output above; the recording matches it (shm_size 4294967296 bytes = 4 GiB).")

recorded output (from the course machine, not this run):
{
 "image": "vllm/vllm-openai-cpu:v0.32.0",
 "ports": [
  [
   "127.0.0.1",
   "8320",
   8000
  ]
 ],
 "shm_size": 4294967296,
 "first_arguments": [
  "Qwen/Qwen2.5-0.5B-Instruct",
  "--served-model-name",
  "brightside"
 ]
}
shm_size in the recording: 4294967296 bytes = 4 GiB


**What you should see.** The image is the pinned tag, never `latest`. The port maps the host's 8320 to the container's 8000, the port vLLM listens on. The shared memory is `4g` in the file, which Docker reports as 4294967296 bytes (4 GiB). Lesson 08 starts its container with the same `--shm-size 4g`. vLLM passes data between worker processes through `/dev/shm`, and Docker's 64 MB default is too small.

### Step 2: configure from the environment, not from the file
The model name is `${SERVE_MODEL:-Qwen/Qwen2.5-0.5B-Instruct}`. That means: use the shell variable `SERVE_MODEL` if it is set, else the default. One value changes without editing the file. Secrets follow the same idea in a different place: a key is set at run time (lesson 16) and never written into the file, because a file is copied, committed and printed.

The cell below changes the model in the environment and resolves the file again. It downloads nothing.

In [5]:
override = dict(os.environ, SERVE_MODEL="Qwen/Qwen2.5-1.5B-Instruct")
res = subprocess.run([*COMPOSE_ARGS, "config", "--format", "json"], capture_output=True, text=True, env=override, cwd=Path(".").resolve())
print("model with SERVE_MODEL set:", json.loads(res.stdout)["services"]["serve"]["command"][0])
live_override = res.returncode == 0

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


If Docker is not running here, the cell above is skipped and the next cell prints the recorded result of the same check.

In [6]:
if not globals().get("live_override", False):
    print("recorded output (from the course machine):", REC["override_model"])

recorded output (from the course machine): Qwen/Qwen2.5-1.5B-Instruct


### Step 3: readiness, the server really starts
Start the service in the background. Then poll its health status every 5 seconds, until Docker reports `healthy` or 480 seconds pass. Docker runs the `healthcheck` from the file, which calls `/health` inside the container.

Two states matter. **Starting:** the process runs, the weights are loading, and `/health` does not answer. **Healthy:** the weights are in memory, `/health` answers 200, and the replica may take traffic. The first state can last minutes; that is the cold start of lesson 31.

The first run downloads the model (about 1 GB) if it is not in your Hugging Face cache. The cache is the folder mounted from `~/.cache/huggingface`, as in lesson 08.

In [7]:
CONTAINER = f"{STACK}-serve-1"
BASE = "http://127.0.0.1:8320"
up = subprocess.run([*COMPOSE_ARGS, "up", "-d"], capture_output=True, text=True, cwd=Path(".").resolve())
if up.returncode != 0:
    raise RuntimeError("compose up failed: " + up.stderr[-300:])
t0, trace, status = time.time(), [], ""
while time.time() - t0 < 480:
    status = subprocess.run(["docker", "inspect", "-f", "{{.State.Health.Status}}", CONTAINER],
                            capture_output=True, text=True).stdout.strip()
    if not trace or trace[-1][1] != status:
        trace.append((round(time.time() - t0), status))
        print(f"t={trace[-1][0]:>4} s  health: {status}")
    if status == "healthy":
        break
    time.sleep(5)
if status != "healthy":
    logs = subprocess.run(["docker", "logs", "--tail", "20", CONTAINER], capture_output=True, text=True)
    raise TimeoutError("not healthy after 480 s: " + logs.stdout[-300:])
import httpx
health = httpx.get(f"{BASE}/health", timeout=5).status_code
models = [m["id"] for m in httpx.get(f"{BASE}/v1/models", timeout=10).json()["data"]]
live_server = True
print("GET /health ->", health, "| served models:", models)

⏭  Skipped: this cell needs [docker,download,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,download,slow.


**What you should see.** One `starting` line, then `healthy` after about 100 s on the course machine (4 CPU cores, model cached). The recording took 96 s and a later run 136 s, so the time varies with the machine. The `/health` answer is 200, and `/v1/models` lists one model, named `brightside`. That name comes from `--served-model-name` in the file. Clients use it, not the file path of the weights, so a model swap later does not change client code.

In [8]:
if not globals().get("live_server", False):
    print("recorded output (from the course machine, not this run):")
    print("health trace (seconds, status):", REC["trace"])
    print("GET /health ->", REC["health"], "| served models:", REC["models"])
else:
    print("live run: healthy after", trace[-1][0], "s; the recording in data/recorded/32.json came from a run like this one.")

recorded output (from the course machine, not this run):
health trace (seconds, status): [[0, 'starting'], [96, 'healthy']]
GET /health -> 200 | served models: ['brightside']


### Step 4: where a GPU pod can run (Kubernetes manifest, read only)
The file `../infra/k8s/brightside-serve.yaml` describes the same server for a Kubernetes cluster: a Deployment, a Service and a PodDisruptionBudget. The placement block decides the node. Four rules matter.

| Rule | In the file | What it does |
| --- | --- | --- |
| GPU type | `nodeSelector: cloud.google.com/gke-accelerator: nvidia-l4` | places the pod only on nodes with that GPU label |
| GPU count | `limits: nvidia.com/gpu: 1` | asks for one GPU; the GPU is an extended resource the device plugin advertises |
| Taint and toleration | the GPU taint `nvidia.com/gpu=present:NoSchedule` on GPU nodes | keeps other pods off GPU nodes; this pod tolerates it |
| Spot pool | `cloud.google.com/gke-spot: "true"` node label, and a toleration for the taint you put on the pool | runs on a cheaper pool that the provider can reclaim |

Two facts about GKE, checked against its docs. **The Spot label is added by GKE**: "GKE adds the `cloud.google.com/gke-spot=true` label to nodes that use Spot VMs." **The Spot taint is one you set** when you create the pool, for example `--node-taints=cloud.google.com/gke-spot="true":NoSchedule --spot`. The pod's toleration matches the taint you put on the pool. (Source: GKE Spot VM docs, `docs.cloud.google.com/kubernetes-engine/docs/how-to/spot-vms`.) **The GPU taint** is added by GKE only when the cluster also has a non-GPU node pool; otherwise you set it yourself (GKE GPU docs, `docs.cloud.google.com/kubernetes-engine/docs/how-to/gpus`).


The cell reads the file and prints those fields, so you see the values the scheduler gets.

In [9]:
import yaml
docs = [d for d in yaml.safe_load_all(Path("../infra/k8s/brightside-serve.yaml").read_text()) if d]
dep = next(d for d in docs if d["kind"] == "Deployment")
pod = dep["spec"]["template"]["spec"]
ctr = pod["containers"][0]
print("kinds in the file:", [d["kind"] for d in docs])
print("nodeSelector:", pod["nodeSelector"])
print("tolerations:", [f'{t["key"]}={t["value"]}:{t["effect"]}' for t in pod["tolerations"]])
print("GPU limit:", ctr["resources"]["limits"])
print("probes:", {k: ctr[k]["httpGet"]["path"] for k in ("startupProbe", "readinessProbe", "livenessProbe")})
sp = ctr["startupProbe"]
print(f"startupProbe: periodSeconds={sp['periodSeconds']}, failureThreshold={sp['failureThreshold']}, "
      f"so it gives up after {sp['failureThreshold'] * sp['periodSeconds']} s")
print("termination grace seconds:", pod["terminationGracePeriodSeconds"])

kinds in the file: ['Deployment', 'Service', 'PodDisruptionBudget']
nodeSelector: {'cloud.google.com/gke-accelerator': 'nvidia-l4', 'cloud.google.com/gke-spot': 'true'}
tolerations: ['nvidia.com/gpu=present:NoSchedule', 'cloud.google.com/gke-spot=true:NoSchedule']
GPU limit: {'nvidia.com/gpu': 1}
probes: {'startupProbe': '/health', 'readinessProbe': '/health', 'livenessProbe': '/health'}
startupProbe: periodSeconds=10, failureThreshold=60, so it gives up after 600 s
termination grace seconds: 120


**What you should see.** The pod asks for one L4 GPU and tolerates the GPU taint. Without the `nvidia.com/gpu` limit, the scheduler places the pod on any node: the server then starts on a CPU and fails or runs without its GPU. Without the node selector, a pod with a GPU limit waits in `Pending` until a matching node exists.

### Step 5: check the manifest offline (kubeconform)
`kubeconform` checks each object against the Kubernetes API schema, without a cluster. Strict mode also rejects unknown fields, which catches a typo such as `replica:`. The schemas for the three kinds in the file are in `infra/schemas/v1.33.0-standalone-strict/`, so the check runs offline. Kinds without a schema here (a custom resource such as a KubeRay or LeaderWorkerSet object) fail the check unless you add their schemas.

**Reading the template.** `SCHEMA_TEMPLATE` is a path pattern that kubeconform fills in for each object: `{{.NormalizedKubernetesVersion}}` becomes `1.33.0`, `{{.StrictSuffix}}` becomes `-strict`, `{{.ResourceKind}}` becomes the lowercase kind (`deployment`) and `{{.KindSuffix}}` becomes the group and version (`-apps-v1`). So the Deployment is checked against `infra/schemas/v1.33.0-standalone-strict/deployment-apps-v1.json`. **The folder layout** is `infra/schemas/` → one folder per Kubernetes version and mode → one JSON file per kind. The paths are relative to the **course root** (`topics/kv-forge-15h/`), the folder above `notebooks/`, so the cell runs kubeconform from there (`cwd=..`). The schemas come from `yannh/kubernetes-json-schema` (Apache-2.0, derived from the Kubernetes OpenAPI spec); the source, licence and checksums are in `data/SOURCES.md`, and the upstream licence is `infra/schemas/LICENSE`.

Install the single binary once: download `kubeconform-linux-amd64.tar.gz` from the release page of `yannh/kubeconform` (version v0.8.0, 2026-06-04) and put `kubeconform` on your PATH. If the binary is missing, the cell prints the recorded result from the course machine instead.

In [10]:
KUBECONFORM = shutil.which("kubeconform")
SCHEMA_TEMPLATE = "infra/schemas/{{.NormalizedKubernetesVersion}}-standalone{{.StrictSuffix}}/{{.ResourceKind}}{{.KindSuffix}}.json"
if KUBECONFORM:
    res = subprocess.run([KUBECONFORM, "-strict", "-summary", "-kubernetes-version", "1.33.0",
                          "-schema-location", SCHEMA_TEMPLATE, "infra/k8s/brightside-serve.yaml"],
                         capture_output=True, text=True, cwd=Path("..").resolve())
    print(res.stdout.strip() or res.stderr.strip())
    live_kc = res.returncode == 0
else:
    print("kubeconform is not on PATH. Recorded output from the course machine:")
    print(REC["kubeconform_ok"])

Summary: 3 resources found in 1 file - Valid: 3, Invalid: 0, Errors: 0, Skipped: 0


**What you should see.** `Valid: 3, Invalid: 0`: the Deployment, the Service and the PodDisruptionBudget each passed the schema. The check does not test the cluster. It tests that the file is a well-formed object, and that is the first gate in a pipeline.

### Step 6: choose where the server runs
Three ways to run the same server:

| Option | You run | Scales to zero? | Start-up cost | Fits | Watch for |
| --- | --- | --- | --- | --- | --- |
| Self-hosted VM with Compose (this lesson) | host, container, updates | no, unless you build it | model load, minutes (Step 3: about 100 s here) | one model, steady load | you own uptime, patches, the GPU driver |
| Kubernetes, GKE GPU node pools | pods, pools, probes | yes, if the autoscaler may go to zero (lesson 31) | node start plus model load | many models and teams; multi-host (lesson 34) | the cost of running a cluster |
| Serverless GPU, Cloud Run, one GPU per instance | the container only | yes; idle instances go to zero | a few seconds for the container (an estimate, not measured in this course), then the model load | bursty traffic, small models | instance-based billing; minimum instances are billed while idle |

Only the Step 3 times are measured. The others are estimates.

Three named tools go further. None is run in this course.
- **vLLM Helm chart.** vLLM's docs have a Helm guide and chart (`examples/deployment/chart-helm` in the vLLM repository). It packages the probes and GPU settings you just read.
- **KubeRay.** The Ray project's Kubernetes operator. vLLM's Kubernetes page lists it for running one model across several nodes.
- **LeaderWorkerSet (kubernetes-sigs/lws).** An API for a group of pods that starts and is replaced as one unit: a leader and its workers. It is the shape that a model split across hosts needs (lesson 34).

On GKE the NVIDIA driver is installed for you on a Google-managed GPU pool. You choose the driver version when you create the pool. Lesson 33 shows what a node loss does to in-flight streams.

## At work
- **Config and routing changes break quality, not just uptime.** Anthropic's postmortem (2025-09-17) describes a routing change that sent short-context requests to servers built for a 1M-token window; spot checks missed it (RESEARCH.md, section 4). → lesson 35.

- **Spot capacity is reclaimed.** Compute Engine can reclaim Spot VMs at any time, so a Spot pool needs a retry path and a PodDisruptionBudget; a node loss ends the streams on it (GKE Spot docs; RESEARCH.md, section 7, row 8). → lesson 33.
- **The ready check must allow for the load.** vLLM's Kubernetes page starts its probes at 60 s and advises raising the failure threshold when startup is slower. A 4-minute load fails a probe that gives up after one minute. → lesson 31 (the cold start) and the Try it below.
- **Go deeper:** the vLLM Helm chart, KubeRay, LeaderWorkerSet and the Gateway API Inference Extension. They are Kubernetes topics, not taught here.

## Common mistakes
**1. A number written as text.** `replicas: "two"` looks right and fails in the cluster. `kubeconform` catches it before that. This cell writes the broken file and runs the same check, so the error prints. **This cell fails on purpose.**

In [11]:
broken = Path("../data/32/broken.yaml")
broken.parent.mkdir(parents=True, exist_ok=True)
broken.write_text(Path("../infra/k8s/brightside-serve.yaml").read_text().replace("replicas: 2", 'replicas: "two"'))
if not KUBECONFORM:                                 # no binary here: raise the recorded error instead
    raise RuntimeError(REC["kubeconform_broken"] + "  [recorded output: kubeconform is not on PATH on this machine]")
res = subprocess.run([KUBECONFORM, "-strict", "-summary", "-kubernetes-version", "1.33.0",
                      "-schema-location", SCHEMA_TEMPLATE, "data/32/broken.yaml"],
                     capture_output=True, text=True, cwd=Path("..").resolve())
if res.returncode != 0:
    raise RuntimeError(res.stdout.strip())

RuntimeError: data/32/broken.yaml - Deployment brightside-serve is invalid: problem validating schema. Check JSON formatting: jsonschema validation failed with 'file://./infra/schemas/v1.33.0-standalone-strict/deployment-apps-v1.json#' - at '/spec/replicas': got string, want null or integer
Summary: 3 resources found in 1 file - Valid: 2, Invalid: 1, Errors: 0, Skipped: 0

**The fix:** write the number as a number (`replicas: 2`), run the check again, and keep it in the pipeline.

**2. A liveness probe that fires before the weights load.** With the defaults in the vLLM page (`initialDelaySeconds: 60`, `failureThreshold` 3, `periodSeconds` 10), a liveness probe can restart the pod after about 90 s, while the weights are still loading. The pod restarts, loads again, and never becomes ready. Put a startup probe in front of both, as the manifest does.

**3. A GPU pod without the GPU request.** It schedules on a CPU node and never gets a GPU. Check `nvidia.com/gpu` in the limits, then `kubectl get pods` for `Pending`.

## Production checklist
- Pin the image tag. Never use `latest`: the lab uses `v0.32.0`, and an upgrade is a reviewed change.
- Keep secrets out of files and out of printed output. A Kubernetes Secret, or the run-time environment, carries the key. Do not paste `docker compose config` output into a ticket if a key is set.
- Set a startup probe long enough for the slowest load you measured (lesson 27 on your image). Readiness and liveness start after it passes.
- Request one GPU per pod with `nvidia.com/gpu`, and select the GPU type with a node label. Check for pods stuck in `Pending`.
- Set `terminationGracePeriodSeconds` longer than the longest generation, and a PodDisruptionBudget, so a drain finishes in-flight streams.
- Run `kubeconform` in the pipeline on every manifest, before it reaches a cluster.

## Try it
The probe gives up after failureThreshold x periodSeconds seconds (plus initialDelaySeconds). The manifest's startup probe has 60 x 10 s = 600 s, and the startup probe has no initialDelaySeconds. The printed line in Step 4 shows these two values.

Your model takes about 12 minutes to load (a stand-in: measure yours, as in lesson 27). The startup probe checks `/health` every 10 seconds. How many failures must it allow, with a 50% margin, so the pod is not killed while it loads? Compute it in the cell, then check your answer.

In [12]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [13]:
import math
load_s, period_s = 12 * 60, 10
needed = math.ceil(load_s / period_s)
print("probes needed for the load:", needed)
print("failureThreshold with a 50% margin:", math.ceil(needed * 1.5))

probes needed for the load: 72
failureThreshold with a 50% margin: 108


## Recap
- Package the server at one pinned version, configure it from the environment, and keep secrets out of files and printed output.
- Readiness, not process start, decides when traffic may reach a replica; the startup probe must allow for the weights to load.
- A GPU pod needs three things: the GPU limit `nvidia.com/gpu`, a node label for the GPU type, and a toleration for the GPU taint. Spot pools add a retry path.
- Validate every manifest offline with `kubeconform` before a cluster sees it. Serverless GPUs trade the cluster for a per-instance cold start.
- **Interview:** "How do you deploy an LLM server on GPUs?" Strong answer: a pinned image, config from the environment, a startup and readiness probe sized to the measured load time, one GPU per pod selected by node label and toleration, a drain that outlasts the longest generation, and manifests validated in CI.

Stop the containers. Without Docker, this cell does nothing.

In [14]:
if globals().get("live_server", False):
    subprocess.run([*COMPOSE_ARGS, "down"], capture_output=True, text=True, cwd=Path(".").resolve())
    print("containers stopped")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.
